# EchoMind · 數據分析第一步：認識 Calm Thinking 數據集

歡迎來到 EchoMind 數據分析工作台。本 Notebook 是為初學 Pandas 與心理學實驗數據處理設計的第一個探索環境。

---

## Section 1 — Introduction (研究背景與基本概念)

### 1.1 研究背景
本數據來自 **Eberle et al. (2024)** 的 *"Calm Thinking"* 研究。該研究是一項大規模網絡認知偏誤修正（Cognitive Bias Modification for Interpretation, CBM-I）隨機對照臨床試驗，旨在探討針對焦慮人群的認知解釋偏誤干預效果。

在本次探索中，我們使用的是該研究的核心評估任務數據：**`rr.csv` (Recognition Ratings, 認知再認評分任務)**。

### 1.2 核心概念辨析
在進行數據分析時，清晰區分以下三個層次是理解任何表格數據的前提：

1. **Participant（受試者／參與者）**：
   - 參與心理學實驗的具體個體。在數據集中由 `participant_id` 唯一識別。
2. **Observation（觀測值／測量記錄）**：
   - 在某個具體時間點或實驗條件下的一次測量事件。在 `rr.csv` 中，**每一行（Row）代表一名參與者在某個特定 Session（實驗階段）的一次完整評分記錄**，而非一名孤立的受試者。
3. **Variable（變量／特徵）**：
   - 所測量的具體屬性或行為指標。在數據集中對應於每一列（Column），例如反應時間（`time_on_page`）、不同情境下的解釋評分（如 `elevator_ns`）等。

### 1.3 本次目標
> **本次 Notebook 的唯一目標**：認識數據結構、檢視維度、熟悉 DataFrame 基本操作。**先不進行數據清洗、篩選、統計推論或建模**。

---
## Section 2 — Import Libraries (載入必備庫)

載入進行數據分析的核心庫：
- `pandas`：Python 最核心的表格數據處理與分析庫，提供高效的 `DataFrame` 數據結構。
- `numpy`：提供底層數值運算支援。
- `pathlib.Path`：物件導向的路徑處理工具，確保在不同作業系統（Windows/macOS/Linux）下均能穩定定位路徑。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

print(f"Pandas 版本: {pd.__version__}")
print(f"NumPy 版本: {np.__version__}")

---
## Section 3 — Load Dataset (讀取數據集)

### 相對路徑處理原則
本 Notebook 存放於 `notebooks/` 子目錄下，而原始數據存放在專案的 `data/raw/rr.csv`。
我們使用 `pathlib.Path` 自動定位專案根目錄，避免硬編碼個人電腦的絕對路徑，確保程式碼具備跨環境可重現性。
若專案目錄內尚未放置檔案，程式會自動搜尋既有的工作區原始存檔作為備援，並提供友好的錯誤提示。

In [ ]:
# 定位專案目錄與數據檔案
current_dir = Path.cwd()
# 若當前在 notebooks 目錄中運行，上一級目錄即為專案根目錄
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

raw_data_path = project_root / "data" / "raw" / "rr.csv"
fallback_source_path = Path(r"C:\Users\Kayla\Documents\Codex\2026-10-28\referenced-chatgpt-conversation-this-is-an-3\work\open-benchmark-discovery-20261030\calm\rr.csv")

if raw_data_path.exists():
    target_file = raw_data_path
    print(f"✓ 找到專案數據文件: {target_file}")
elif fallback_source_path.exists():
    target_file = fallback_source_path
    print(f"ℹ 專案內尚未建立副本，自動從原始研究存檔讀取: {target_file}")
else:
    raise FileNotFoundError(
        f"【檔案讀取錯誤】找不到 rr.csv 數據集！\n"
        f"請確認以下路徑是否存在：\n"
        f"1. 專案目錄: {raw_data_path}\n"
        f"2. 原始存檔: {fallback_source_path}"
    )

# 讀取 CSV 數據集為 DataFrame
df = pd.read_csv(target_file)
print(f"✓ 成功載入數據！DataFrame 維度 (Rows, Columns): {df.shape}")

---
## Section 4 — Inspect Dataset (檢視數據結構)

在載入任何數據集後，通常應依次執行以下四個基本檢查，快速建立對數據結構的全貌認識。

### 4.1 檢視維度 (`df.shape`)
執行目的：確認 DataFrame 的行列規模（Row 數量代表總觀測數，Column 數量代表變量數）。

In [ ]:
df.shape

### 4.2 預覽前五筆資料 (`df.head()`)
執行目的：查看各欄位的實際資料範例、編碼樣式以及是否包含預期的識別字。

In [ ]:
df.head()

### 4.3 列出所有欄位名稱 (`df.columns.tolist()`)
執行目的：檢查全部 45 個變量的命名規範，確認情境變量（如 `blood_test_*`、`elevator_*`）與元數據欄位。

In [ ]:
df.columns.tolist()

### 4.4 檢視欄位資訊與資料型態 (`df.info()`)
執行目的：檢查各欄位的資料型別（int64, float64, object）及非空值（Non-Null）數量，確認記憶體使用概況。

In [ ]:
df.info()

---
## Section 5 — Understanding the Rows & Items (解讀行與變量含義)

根據 Eberle et al. (2024) 原始文獻與 Codebook，我們對當前數據結構梳理出以下**已獲核驗的事實**：

### 5.1 行（Rows）的含義：測量事件（Measurement Occasions）
- **每行數據代表一次測量事件，而非單獨一名受試者**。
- 數據集中包含受試者編號：`participant_id`（共 1,524 位獨立受試者）。
- **同一位受試者可能在數據中出現多次**（縱向重複測量）：
  - `preTest`（基線前測）：1,524 筆
  - `thirdSession`（第 3 次干預）：693 筆
  - `fifthSession`（第 5 次干預）：558 筆
  - `PostFollowUp`（追蹤評估）：484 筆
  - 全表總計 3,259 筆觀測值。同一個 `participant_id` 在同一 `session_only` 中無重複記錄。

### 5.2 評分欄位（Rating Columns）結構
數據集中包含 **36 個評分欄位**，源自 9 個模糊日常情境，每個情境評估 4 種類型的解釋：
- `_ns` (**Negative Threat-Specific**)：負面威脅特定解釋（CBM-I 核心靶向指標）。
- `_ps` (**Positive Threat-Specific**)：正面威脅特定解釋（CBM-I 核心靶向指標）。
- `_nf` (**Negative Threat-General / Filler**)：負面非威脅填充題。
- `_pf` (**Positive Threat-General / Filler**)：正面非威脅填充題。

9 個情境主題分別為：`blood_test`（抽血）、`elevator`（電梯）、`job`（工作）、`lunch`（午餐）、`meeting_friend`（會友）、`noise`（異響）、`scrape`（擦傷）、`shopping`（購物）、`wedding`（婚禮）。

### 5.3 評分尺度與心理學意義
- **有效評分區間**：1 到 4 分（4-point Likert Scale）：
  - `1` = Very unlike what I was thinking（非常不符合我的想法）
  - `2` = Somewhat unlike what I was thinking（有些不符合我的想法）
  - `3` = Somewhat like what I was thinking（有些符合我的想法）
  - `4` = Very like what I was thinking（非常符合我的想法）
- **特殊缺失值編碼**：**`555`** 代表缺失值（Skipped / Prefer not to answer）。
  - 全表共記錄 761 個 `555` 評分單元格。
  - **重要警示**：後續計算均值或統計指標時，必須先將 `555` 轉換為 `NaN`，切勿直接作為數值 555 參與均值計算。
- **心理學概念邊界**：
  - 本評分反映的是**「情境語義相似度與解釋偏誤傾向」**，不能擅自等同於臨床診斷、社交焦慮量表或人際拒絕預期。

---

### 5.4 思考問題（供後續分析自行思考）
> 💭 **問題 1**：當我們想要評估「人類受試者的基線解釋偏誤」時，應該選取 `session_only == 'preTest'` 還是全量數據？為什麼？
>
> 💭 **問題 2**：`elevator_ns`（負面特定）與 `elevator_ps`（正面特定）的評分差距，如何反映個體的解釋偏誤程度？
>
> 💭 **問題 3**：如果將這 9 個情境作為 LLM 的 Prompt，我們應該如何設計 Prompt 與評估標準，才能與人類受試者的評分分佈進行對齊？

---
## Section 6 — Next Steps (後續學習規劃)

在完成第一步的環境驗證與結構認識後，後續的數據探索步驟規劃如下（本次不執行具體運算）：

1. **Missing Values（缺失值處理）**：
   - 將 `555` 數值替換為 `np.nan`。
   - 統計各情境變量的缺失比率，分析是否有特定題目缺失率偏高。
2. **Duplicates & Repeated Measurements（重複測量結構）**：
   - 驗證每位受試者在各 Session 的留存率與流失情況（Attrition Analysis）。
   - 篩選 `preTest` 作為基線單一觀測子集。
3. **Descriptive Statistics（描述性統計）**：
   - 計算各情境中 `ns`, `ps`, `nf`, `pf` 的均值與標準差。
   - 檢驗前測基線的內部一致性信度（Cronbach's $\alpha$）。
4. **Score Distributions & Visualization（評分分佈與視覺化）**：
   - 繪製正面與負面解釋評分的頻數分佈直方圖與箱型圖。
5. **Benchmark Construction（基準構建）**：
   - 將 9 個情境的原始刺激文本與 36 項解釋選項整理成 JSON/YAML 測試集，供 EchoMind 的 LLM Context Injection 實驗評測。